# Repair RQ3: predicted spans, and stage A with and without KEEP training

How much does repair degrade when Channel A, not the rule checker, decides which claims to edit, and does KEEP training (stage A's falsely flagged items) limit the harm? Trains stage A again without KEEP items, then evaluates both models and the rule repairer with Channel A's spans on the same 500 tune-track items. Attach the stage A notebook's output and the faithguard-detector-models dataset. Expect about 7.5 GPU hours.

**On Kaggle:** Create → New Notebook → File → Import Notebook → this file. Settings: *Accelerator* **GPU T4 x2**, *Internet* **on**.
For long jobs use **Save Version → Save & Run All (Commit)**: it runs in the background for up to 12 hours and keeps everything in `/kaggle/working/outputs`.
To continue in a later notebook, add this notebook's output as input (*Add Input → Your Work*).

**Afterwards:** download `outputs/*/results.json` and `report.md` files into `runs/` in the repository, and log the session's GPU hours in `logs/gpu-hours.csv`. Checkpoints are saved regularly; re-running the same cell resumes from the latest one.

In [ ]:
# Setup: clone FaithGuard, install it with the pinned GPU packages, fetch the data.
import shutil, socket
try:
    socket.gethostbyname("pypi.org")
except OSError:
    raise RuntimeError("No internet in this session. Kaggle: verify your phone number (Settings), then switch Internet on in the right-hand panel.")
if not shutil.which("nvidia-smi"):
    raise RuntimeError("No GPU in this session. Kaggle: verify your phone number (Settings), then set Accelerator to GPU T4 x2. Colab: Runtime > Change runtime type > T4 GPU.")
import os, subprocess, sys
WORK = '/kaggle/working' if os.path.exists('/kaggle') else '/content'
os.chdir(WORK)
if not os.path.exists('faithguard'):
    subprocess.check_call(['git', 'clone', '--depth', '1', 'https://github.com/SharafThawfeek/FaithGurd-new.git', 'faithguard'])
os.chdir('faithguard')
REQUIREMENTS = ['transformers==5.19.0', 'peft==0.21.2', 'accelerate==1.15.0', 'bitsandbytes==0.50.2', 'huggingface_hub==1.33.0', 'datasets==5.1.0', 'lettucedetect==0.2.3']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *REQUIREMENTS])
# Kaggle's image ships torchao 0.10, which PEFT 0.21 rejects when it adds LoRA to fp16 weights; nothing here uses it
subprocess.call([sys.executable, '-m', 'pip', 'uninstall', '-y', '-q', 'torchao'])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'])
# this kernel started before the install, and the clone's own folder would shadow the package: import from src/
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
OUT = os.path.join(WORK, 'outputs'); os.makedirs(OUT, exist_ok=True)
!faithguard data download all
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Stage A's adapter and Channel A's model

In [ ]:
import glob, os
for flat in glob.glob('/kaggle/input/**/*__model__*', recursive=True):  # the models dataset, folders flattened
    target = os.path.join(WORK, 'prev', *os.path.basename(flat).split('__'))
    os.makedirs(os.path.dirname(target), exist_ok=True)
    if not os.path.exists(target):
        os.symlink(flat, target)
CHANNEL_A = os.path.join(WORK, 'prev', 'channel-a', 'model')
ADAPTER = sorted(glob.glob('/kaggle/input/**/repair-sft/adapter', recursive=True))[0]
print(CHANNEL_A, os.path.exists(CHANNEL_A), ADAPTER)
MODE = 'fp16'

## Stage A without KEEP items (no falsely flagged claims in training)

In [ ]:
!faithguard sft --mode program --false-flags 0 --out runs/sft-nokeep
!python -m faithguard.train.repair_sft --data runs/sft-nokeep/repair-program-train.jsonl --out {OUT}/repair-sft-nokeep --mode $MODE

## Evaluate with Channel A's spans (and the no-KEEP model with the rule checker's, for reference)

In [ ]:
!python -m faithguard.train.repair_eval --adapter $ADAPTER --load $MODE --spans $CHANNEL_A --out {OUT}/eval-sft-channel-a-spans
!python -m faithguard.train.repair_eval --adapter {OUT}/repair-sft-nokeep/adapter --load $MODE --spans $CHANNEL_A --out {OUT}/eval-nokeep-channel-a-spans
!python -m faithguard.train.repair_eval --adapter {OUT}/repair-sft-nokeep/adapter --load $MODE --out {OUT}/eval-nokeep-rule-spans
!python -m faithguard.train.repair_eval --mode rules --spans $CHANNEL_A --out {OUT}/eval-rules-channel-a-spans

## Results

In [ ]:
import glob, json
for path in sorted(glob.glob(OUT + '/**/results.json', recursive=True)):
    print(path)
    print(json.dumps({k: v for k, v in json.load(open(path)).items() if k in ('summary', 'channel_a', 'channel_b', 'auroc', 'false_alarm_clean', 'wrong_context_recall')}, indent=1)[:1500])